In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

for rel_path in ['../', '../../', '../../../']:
    abs_path = os.path.abspath(rel_path)
    if abs_path not in sys.path:
        sys.path.insert(0, abs_path)

from benchmarking.utils import read_perf_eval_json_files

COLOR_MAP    = {'Encrypted': '#EF553B', 'Non-Encrypted': '#636EFA'}
SYMBOL_MAP   = {'Encrypted': 'circle',  'Non-Encrypted': 'square'}
CONC_ORDER   = [1, 3, 5, 8, 16, 32]
CONC_STR     = [str(c) for c in CONC_ORDER]
INPUT_ORDER  = [3900, 8000, 16000]
INPUT_LABELS = [f'{n:,} input tokens' for n in INPUT_ORDER]

print('Setup complete')

# DataKrypto TEE Performance Analysis
## Encrypted vs Non-Encrypted Model — Benchmarking Report

---

### Architecture

```
Non-Encrypted:
  Client ──────────────────────────► SambaNova Inference Server
                    (1 network hop)

Encrypted:
  Client ──► DataKrypto TEE ───────► SambaNova Inference Server
              │  encrypts input          returns encrypted chunks
              │  decrypts each output ◄─────────────────────────┘
              │  chunk (streaming)
              └──► Client
              (2 network hops total)
```

### Models
| Model | Description |
|-------|-------------|
| `Llama-xLAM-2-8b-fc-r` | Base model (plain inference) |
| `Llama-xLAM-2-8b-fc-r-encrypted` | Runs through DataKrypto TEE |

> Note: The two models are served by **separate SambaNova endpoints** which may have
> different hardware allocations or decode configurations. Any baseline performance
> difference between them is an endpoint-level characteristic — not caused by the TEE.

### Test Matrix
- **Input tokens**: 3,900 / 8,000 / 16,000 (varied — primary axis of analysis)
- **Output tokens**: 100 (fixed)
- **Concurrencies**: 1, 3, 5, 8, 16, 32
- Each concurrency level = **N separate HTTP requests sent simultaneously** (not a single batch API call — N independent requests hitting the server at the same time, which the server can dynamically batch together for inference)
- **Multi Run (10x)**: 10 sequential rounds of N simultaneous requests — provides statistical robustness and tests sustained server/TEE behavior over time

> All plots in this report are **faceted by Input Tokens** so encryption / network / server-side
> behavior can be compared directly across prompt sizes.

### TEE Overhead Sources
1. **Input Encryption** — encrypts the prompt (grows with input token count)
2. **Output Decryption** — decrypts each streaming token chunk (grows with output tokens)
3. **Network Hop** — extra Client → TEE → Server roundtrip (dominant overhead)

> **TEE capacity constraint**: The TEE has **8 cores** and processes encryption/decryption
> sequentially per core. At concurrency > 8, the TEE queue fills up causing requests to wait,
> which means concurrent requests may arrive at the inference server **staggered** rather than
> simultaneously — reducing the effective batch size the server sees.

---

In [ ]:
DATA_ROOT = os.path.abspath('../../data/datakrypto')

DATASETS = {
    'enc_multiple': {
        'consolidated': f'{DATA_ROOT}/high_ss/speed_bench_test_encrypted_multiple/consolidated_results/20260506-120053.565163-complete.xlsx',
        'run_dir':      f'{DATA_ROOT}/high_ss/speed_bench_test_encrypted_multiple/20260506-120053.565163-complete',
        'encrypted': True,
        'run_type': 'Multi Run (10x)',
        'label': 'Encrypted',
    },
    'plain_multiple': {
        'consolidated': f'{DATA_ROOT}/high_ss/speed_bench_test_nonencrypted_multiple/consolidated_results/20260506-121009.676284.xlsx',
        'run_dir':      f'{DATA_ROOT}/high_ss/speed_bench_test_nonencrypted_multiple/20260506-121009.676284',
        'encrypted': False,
        'run_type': 'Multi Run (10x)',
        'label': 'Non-Encrypted',
    },
}

dfs = []
for key, meta in DATASETS.items():
    df = pd.read_excel(meta['consolidated'])
    df['dataset_key'] = key
    df['encrypted']   = meta['encrypted']
    df['run_type']    = meta['run_type']
    df['label']       = meta['label']
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# Derived columns — input-token labels are now the primary facet
df_all['in_tokens_label']     = df_all['num_input_tokens'].apply(lambda n: f'{int(n):,} input tokens')
df_all['out_tokens_label']    = df_all['num_output_tokens'].astype(str) + ' output tokens'
df_all['total_requests']      = df_all['num_completed_requests'] + df_all['number_errors']
df_all['error_rate_pct']      = (df_all['number_errors'] / df_all['total_requests'].replace(0, np.nan)) * 100
df_all['completion_rate_pct'] = 100 - df_all['error_rate_pct']

# Ordered categoricals for consistent facet ordering
df_all['in_tokens_label'] = pd.Categorical(
    df_all['in_tokens_label'], categories=INPUT_LABELS, ordered=True
)

print(f'Loaded {len(df_all)} rows across {df_all["dataset_key"].nunique()} datasets')
print(f'Input tokens: {sorted(df_all["num_input_tokens"].unique())}')
print(f'Output tokens: {sorted(df_all["num_output_tokens"].unique())}')
print(f'Concurrencies: {sorted(df_all["num_concurrent_requests"].unique())}')
df_all.groupby(['label', 'in_tokens_label'])[['num_concurrent_requests']].count().rename(
    columns={'num_concurrent_requests': 'scenarios'}
)

## 1. Dataset Overview

In [ ]:
cols = ['label',
        'num_input_tokens',
        'num_output_tokens',
        'num_concurrent_requests',
        'num_completed_requests',
        'number_errors',
        'error_rate_pct'
        ]
overview = df_all[cols].copy()
overview.columns = ['Model',
                    'Input Tokens',
                    'Output Tokens',
                    'Concurrency',
                    'Completed',
                    'Errors',
                    'Error Rate (%)'
                    ]
overview['Error Rate (%)'] = overview['Error Rate (%)'].fillna(0).round(1)
overview = overview.sort_values(['Input Tokens', 'Model', 'Concurrency'])

def _highlight_errors(row):
    if row['Errors'] > 0:
        return ['background-color: #ffdddd'] * len(row)
    return [''] * len(row)

overview.style.apply(_highlight_errors, axis=1).format({'Error Rate (%)': '{:.1f}%'})

## 2. Error & Completion Rate Analysis

The TEE has **8 cores** processing encryption/decryption sequentially.
At concurrency > 8, the TEE queue fills up and requests may time out or be rejected,
producing errors on the encrypted model only. The plain model should show zero errors.

In [ ]:
fig = px.bar(
    df_all,
    x='num_concurrent_requests',
    y='error_rate_pct',
    color='label',
    facet_col='in_tokens_label',
    barmode='group',
    color_discrete_map=COLOR_MAP,
    category_orders={
        'num_concurrent_requests': CONC_ORDER,
        'in_tokens_label': INPUT_LABELS,
    },
    labels={
        'num_concurrent_requests': 'Concurrency',
        'error_rate_pct':          'Error Rate (%)',
        'label':                   'Model',
        'in_tokens_label':         'Input Tokens',
    },
    title=(
        'Request Error Rate by Concurrency — faceted by Input Tokens<br>'
        '<sub>Errors appear on Encrypted model at concurrency > 8 (TEE queue saturation). '
        'Non-Encrypted should be 0% throughout.</sub>'
    ),
)
fig.update_layout(height=420, width=1200, template='plotly_white')
fig.update_xaxes(type='category')
fig.show()

In [ ]:
# Completed requests per minute — real system throughput capacity
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='num_completed_requests_per_min',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':        'Concurrency (log scale)',
        'num_completed_requests_per_min': 'Completed Requests / min',
        'label':                          'Model',
        'in_tokens_label':                'Input Tokens',
    },
    title=(
        'System Throughput: Completed Requests per Minute — faceted by Input Tokens<br>'
        '<sub>Indicator of system capacity. '
        'Encrypted throughput is increasingly limited by TEE queueing as input tokens grow '
        '(more bytes to encrypt) and at higher concurrencies.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

## 3. Server-Side Metrics — Behavior Across Input Sizes

Server-side metrics measure what happens **inside the SambaNova engine**, after the TEE has
relayed the request. They tell us whether the TEE itself slows the model — and they isolate
the **input-size effect** on prefill vs. decode.

**What to look for as input tokens grow (3,900 → 8,000 → 16,000):**
- **Server TTFT** rises (prefill scales with input length)
- **Server tok/s** for decode should be roughly stable (decode rate is mostly independent
  of input length once prefill is done)
- **Server E2E latency** rises mainly because of prefill, not because of the TEE

**TEE de-batching effect:** because the TEE processes encryption sequentially per core,
concurrent requests reach the inference server staggered. This often keeps the encrypted path's
**effective server batch size smaller**, so its server-side TTFT can be lower than the
non-encrypted path at high concurrency.

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='server_ttft_s_p50',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests': 'Concurrency (log scale)',
        'server_ttft_s_p50':       'Server TTFT p50 (s)',
        'label':                   'Model',
        'in_tokens_label':         'Input Tokens',
    },
    title=(
        'Server Time to First Token — p50 (faceted by Input Tokens)<br>'
        '<sub>TTFT grows with input size (longer prefill). Watch for the de-batching effect: '
        'at high concurrency, encrypted server TTFT can be lower than non-encrypted because '
        'staggered TEE arrivals shrink the effective server batch.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='server_output_token_per_s_p50',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':       'Concurrency (log scale)',
        'server_output_token_per_s_p50': 'Server Output Tok/s p50 (per request)',
        'label':                         'Model',
        'in_tokens_label':               'Input Tokens',
    },
    title=(
        'Server Output Throughput per Request — p50 (faceted by Input Tokens)<br>'
        '<sub>Decode rate should be roughly independent of input size. Any large gap between '
        'encrypted and non-encrypted in the same facet would point at endpoint configuration '
        'differences, not the TEE itself.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='server_end_to_end_latency_s_p50',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':         'Concurrency (log scale)',
        'server_end_to_end_latency_s_p50': 'Server E2E Latency p50 (s)',
        'label':                           'Model',
        'in_tokens_label':                 'Input Tokens',
    },
    title='Server End-to-End Latency — p50 (faceted by Input Tokens)',
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
# Server metrics delta table — Encrypted minus Non-Encrypted (per Input Tokens × Concurrency)
DIMS = ['num_input_tokens', 'num_output_tokens', 'num_concurrent_requests']

df_enc_s   = df_all[df_all['label'] == 'Encrypted'].set_index(DIMS)
df_plain_s = df_all[df_all['label'] == 'Non-Encrypted'].set_index(DIMS)

server_metrics = [
    ('server_ttft_s_p50',              'Server TTFT p50 (s)'),
    ('server_output_token_per_s_p50',  'Server Tok/s p50'),
    ('server_end_to_end_latency_s_p50','Server E2E p50 (s)'),
]

delta_rows = []
for col, label in server_metrics:
    if col not in df_enc_s.columns:
        continue
    for idx in df_enc_s.index:
        if idx not in df_plain_s.index:
            continue
        enc_val   = df_enc_s.loc[idx, col]
        plain_val = df_plain_s.loc[idx, col]
        if pd.isna(enc_val) or pd.isna(plain_val):
            continue
        delta   = enc_val - plain_val
        pct_chg = delta / plain_val * 100 if plain_val != 0 else np.nan
        delta_rows.append({
            'Metric':        label,
            'Input Tokens':  idx[0],
            'Output Tokens': idx[1],
            'Concurrency':   idx[2],
            'Non-Encrypted': plain_val,
            'Encrypted':     enc_val,
            'Delta':         delta,
            '% Change':      pct_chg,
        })

df_server_delta = pd.DataFrame(delta_rows)
df_server_delta = df_server_delta.sort_values(['Metric', 'Input Tokens', 'Concurrency'])

def _pct_color(val):
    if pd.isna(val): return ''
    if val >  50: return 'background-color: #ffaaaa'
    if val >  20: return 'background-color: #ffd8aa'
    if val >   0: return 'background-color: #fffacc'
    if val < -20: return 'background-color: #aaffaa'
    return ''

df_server_delta.style \
    .applymap(_pct_color, subset=['% Change']) \
    .format({
        'Non-Encrypted': '{:.4f}',
        'Encrypted':     '{:.4f}',
        'Delta':         '{:+.4f}',
        '% Change':      '{:+.1f}%',
    })

## 4. TEE Overhead Decomposition (Encrypted Model Only)

Three measurable components are returned by the TEE per request:

| Component | Description | Expected Behavior vs Input Tokens |
|-----------|-------------|----------------------------------|
| **Input Encryption** | Time to encrypt the prompt before sending to server | grows with input token count |
| **Output Decryption** | Cumulative time to decrypt all streaming output chunks | ~constant (output is fixed at 100) |
| **Network Hop** | Round-trip latency through the TEE (Client→TEE→Server→TEE→Client) | grows with concurrency (TEE queue) and payload size |

> Faceting by input tokens lets us see how much of the TEE overhead is driven by **prompt size**
> as opposed to concurrency.

In [ ]:
df_tee = df_all[df_all['label'] == 'Encrypted'].copy()

# Convert ms → s for consistent units with latency charts
df_tee['Network Hop']        = df_tee['server_network_latency_ms_mean'] / 1000
df_tee['Input Encryption']   = df_tee['total_encryption_time_ms_mean']  / 1000
df_tee['Output Decryption']  = df_tee['total_decryption_time_ms_mean']  / 1000

TEE_ID_COLS  = ['in_tokens_label', 'num_concurrent_requests']
TEE_VAL_COLS = ['Network Hop', 'Input Encryption', 'Output Decryption']

df_tee_melt = df_tee[TEE_ID_COLS + TEE_VAL_COLS].melt(
    id_vars=TEE_ID_COLS, var_name='Component', value_name='Overhead (s)'
)

COMP_COLORS = {
    'Network Hop':       '#FF7F0E',
    'Input Encryption':  '#D62728',
    'Output Decryption': '#2CA02C',
}

fig = px.bar(
    df_tee_melt,
    x='num_concurrent_requests',
    y='Overhead (s)',
    color='Component',
    facet_col='in_tokens_label',
    barmode='stack',
    color_discrete_map=COMP_COLORS,
    category_orders={
        'num_concurrent_requests': CONC_ORDER,
        'in_tokens_label': INPUT_LABELS,
    },
    labels={
        'num_concurrent_requests': 'Concurrency',
        'in_tokens_label':         'Input Tokens',
    },
    title=(
        'TEE Overhead Breakdown per Request — faceted by Input Tokens<br>'
        '<sub>Network Hop dominates. Encryption grows with input size (more bytes to encrypt). '
        'Decryption is roughly constant since output tokens are fixed at 100.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(type='category')
fig.show()

In [ ]:
# Network latency growth — most important overhead component
fig = px.line(
    df_tee,
    x='num_concurrent_requests',
    y='server_network_latency_ms_mean',
    color='in_tokens_label',
    facet_col='in_tokens_label',
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':        'Concurrency (log scale)',
        'server_network_latency_ms_mean': 'TEE Network Latency — Mean (ms)',
        'in_tokens_label':                'Input Tokens',
    },
    title=(
        'TEE Network Latency vs Concurrency — faceted by Input Tokens<br>'
        '<sub>Grows with concurrency (TEE queueing) AND with input tokens '
        '(larger encrypted payload to ship).</sub>'
    ),
)
fig.update_layout(height=420, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
# Encryption and decryption times detail — one line per Input Tokens
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Input Encryption Time (ms)', 'Output Decryption Time (ms)'],
)

_input_colors = {3900: '#1f77b4', 8000: '#2ca02c', 16000: '#ff7f0e'}
_shown        = set()

for in_tok, color in _input_colors.items():
    sub  = df_tee[df_tee['num_input_tokens'] == in_tok].sort_values('num_concurrent_requests')
    name = f'{in_tok:,} input tokens'
    show = name not in _shown
    kw   = dict(x=sub['num_concurrent_requests'], mode='lines+markers',
                name=name, line=dict(color=color),
                legendgroup=name, showlegend=show)
    fig.add_trace(go.Scatter(y=sub['total_encryption_time_ms_mean'], **kw), row=1, col=1)
    kw['showlegend'] = False
    fig.add_trace(go.Scatter(y=sub['total_decryption_time_ms_mean'], **kw), row=1, col=2)
    _shown.add(name)

fig.update_xaxes(
    type='log', tickvals=CONC_ORDER, ticktext=CONC_STR,
    title_text='Concurrency (log scale)',
)
fig.update_yaxes(title_text='Time (ms)', col=1)
fig.update_yaxes(title_text='Time (ms)', col=2)
fig.update_layout(
    height=420, width=1200, template='plotly_white',
    legend_title='Input Tokens',
    title=(
        'TEE Encryption & Decryption Times — by Input Tokens<br>'
        '<sub>Encryption scales with input size (more bytes per prompt). '
        'Decryption is roughly flat across input sizes since output is fixed at 100 tokens.</sub>'
    ),
)
fig.show()

## 5. Client-Side Metrics — End-User Experience

Client metrics include the full roundtrip: **Client → (TEE) → Server → (TEE) → Client**.

The gap between encrypted and non-encrypted client metrics is the
**observable user-facing cost** of the TEE encryption layer.

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_ttft_s_p50',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests': 'Concurrency (log scale)',
        'client_ttft_s_p50':       'Client TTFT p50 (s)',
        'label':                   'Model',
        'in_tokens_label':         'Input Tokens',
    },
    title=(
        'Client Time to First Token — p50 (faceted by Input Tokens)<br>'
        '<sub>Both models slow down as input grows (longer prefill). '
        'Encrypted carries TEE network + encryption overhead on top — gap typically widens '
        'with concurrency, but de-batching can flip it at high concurrency on smaller prompts.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
# Client TTFT overhead attribution: stacked TEE components vs total delta
DIMS = ['num_input_tokens', 'num_output_tokens', 'num_concurrent_requests']

df_enc_a   = df_all[df_all['label'] == 'Encrypted'].copy()
df_plain_a = df_all[df_all['label'] == 'Non-Encrypted'][DIMS + ['client_ttft_s_p50']].rename(
    columns={'client_ttft_s_p50': 'client_ttft_plain'}
)
df_attr = df_enc_a.merge(df_plain_a, on=DIMS)

df_attr['client_ttft_delta']    = df_attr['client_ttft_s_p50'] - df_attr['client_ttft_plain']
df_attr['Network Hop (s)']      = df_attr['server_network_latency_ms_mean'] / 1000
df_attr['Input Encryption (s)'] = df_attr['total_encryption_time_ms_mean']  / 1000
df_attr['Output Decryption (s)']= df_attr['total_decryption_time_ms_mean']  / 1000
df_attr['Residual (s)']         = (
    df_attr['client_ttft_delta']
    - df_attr['Network Hop (s)']
    - df_attr['Input Encryption (s)']
    - df_attr['Output Decryption (s)']
).clip(lower=0)

ATTR_COLS   = ['Network Hop (s)', 'Input Encryption (s)', 'Output Decryption (s)', 'Residual (s)']
ATTR_COLORS = {
    'Network Hop (s)':       '#FF7F0E',
    'Input Encryption (s)':  '#D62728',
    'Output Decryption (s)': '#2CA02C',
    'Residual (s)':          '#9467BD',
}

df_attr_melt = df_attr.melt(
    id_vars=['in_tokens_label', 'num_concurrent_requests', 'client_ttft_delta'],
    value_vars=ATTR_COLS, var_name='Component', value_name='Overhead (s)',
).dropna(subset=['Overhead (s)'])

fig = px.bar(
    df_attr_melt,
    x='num_concurrent_requests',
    y='Overhead (s)',
    color='Component',
    facet_col='in_tokens_label',
    barmode='stack',
    color_discrete_map=ATTR_COLORS,
    category_orders={
        'num_concurrent_requests': CONC_ORDER,
        'in_tokens_label': INPUT_LABELS,
    },
    labels={
        'num_concurrent_requests': 'Concurrency',
        'in_tokens_label':         'Input Tokens',
    },
    title=(
        'Client TTFT Overhead Attribution: Encrypted − Non-Encrypted Delta — by Input Tokens<br>'
        '<sub>Stacked bars = measured TEE components. Total bar height = Enc TTFT − Plain TTFT. '
        'Residual = delta not fully explained by TEE metrics (measurement noise / endpoint diff).</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(type='category')
fig.show()

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_end_to_end_latency_s_p50',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':         'Concurrency (log scale)',
        'client_end_to_end_latency_s_p50': 'Client E2E Latency p50 (s)',
        'label':                           'Model',
        'in_tokens_label':                 'Input Tokens',
    },
    title=(
        'Client End-to-End Latency — p50 (faceted by Input Tokens)<br>'
        '<sub>Full request duration as observed by the client. '
        'Both prefill (input size) and TEE queueing (concurrency) push this up.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
# Per-request client generation speed
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_output_token_per_s_p50',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':       'Concurrency (log scale)',
        'client_output_token_per_s_p50': 'Client Output Tok/s p50 (per request)',
        'label':                         'Model',
        'in_tokens_label':               'Input Tokens',
    },
    title=(
        'Client Output Throughput per Request — p50 (faceted by Input Tokens)<br>'
        '<sub>After the first token, subsequent tokens stream at this rate. '
        'Encrypted is bounded by TEE streaming decryption; '
        'Non-Encrypted by server generation at high concurrency.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
# Total system throughput — aggregate across all concurrent requests
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_total_output_throughput',
    color='label',
    symbol='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    category_orders={'in_tokens_label': INPUT_LABELS},
    labels={
        'num_concurrent_requests':        'Concurrency (log scale)',
        'client_total_output_throughput': 'Total Output Throughput (tok/s)',
        'label':                          'Model',
        'in_tokens_label':                'Input Tokens',
    },
    title=(
        'Total Client Output Throughput — All Concurrent Requests (faceted by Input Tokens)<br>'
        '<sub>Non-Encrypted scales nearly linearly with concurrency. '
        'Encrypted ceiling drops as input tokens grow because each request takes longer to '
        'encrypt + ship through the TEE.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

## 6. Per-Request Distributions

Box plots using individual request data from all 10 rounds, faceted by **Input Tokens**.
Error requests (where `error_code` is set) are excluded.

In [ ]:
def _parse_fname(fname):
    """Extract (input_tokens, output_tokens, concurrency) from benchmark filename.

    Filename format:
        synthetic_<idx>_<model>_<input_tokens>_<output_tokens>_<concurrency>_stream_<uuid>_*
    """
    base  = os.path.basename(str(fname)).replace('_individual_responses.json', '')
    parts = base.split('_')
    try:
        return int(parts[3]), int(parts[4]), int(parts[5])
    except (IndexError, ValueError):
        return None, None, None

dfs_ind = []
for key, meta in DATASETS.items():
    try:
        df_ind = read_perf_eval_json_files(meta['run_dir'], type='individual_responses')
        if df_ind.empty:
            continue
        df_ind['label']    = meta['label']
        df_ind['run_type'] = meta['run_type']
        parsed = df_ind['filename'].apply(lambda f: pd.Series(_parse_fname(f)))
        df_ind['num_input_tokens']        = parsed[0]
        df_ind['num_output_tokens']       = parsed[1]
        df_ind['num_concurrent_requests'] = parsed[2]
        dfs_ind.append(df_ind)
    except Exception as e:
        print(f'Warning: could not load {key}: {e}')

df_individual = pd.concat(dfs_ind, ignore_index=True)
df_individual = df_individual.dropna(
    subset=['num_input_tokens', 'num_output_tokens', 'num_concurrent_requests', 'client_ttft_s']
)
df_individual['num_input_tokens']        = df_individual['num_input_tokens'].astype(int)
df_individual['num_output_tokens']       = df_individual['num_output_tokens'].astype(int)
df_individual['num_concurrent_requests'] = df_individual['num_concurrent_requests'].astype(int)
df_individual['in_tokens_label']         = df_individual['num_input_tokens'].apply(lambda n: f'{n:,} input tokens')
df_individual['concurrency_str']         = df_individual['num_concurrent_requests'].astype(str)
df_individual['is_error']                = df_individual['error_code'].notna()
df_individual['in_tokens_label'] = pd.Categorical(
    df_individual['in_tokens_label'], categories=INPUT_LABELS, ordered=True
)

print(f'Loaded {len(df_individual):,} individual request records')
df_individual.groupby(['label', 'in_tokens_label', 'is_error'])[['client_ttft_s']].count().rename(
    columns={'client_ttft_s': 'count'}
)

In [ ]:
df_box = df_individual[~df_individual['is_error']].copy()

fig = px.box(
    df_box,
    x='concurrency_str',
    y='client_ttft_s',
    color='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    category_orders={
        'concurrency_str': CONC_STR,
        'in_tokens_label': INPUT_LABELS,
    },
    points='outliers',
    labels={
        'concurrency_str':  'Concurrency',
        'client_ttft_s':    'Client TTFT (s)',
        'label':            'Model',
        'in_tokens_label':  'Input Tokens',
    },
    title=(
        'Client TTFT Distribution per Concurrency — faceted by Input Tokens<br>'
        '<sub>Encrypted shows higher median AND wider spread due to variable TEE network latency. '
        'Spread grows with both input size and concurrency.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.show()

In [ ]:
fig = px.box(
    df_box,
    x='concurrency_str',
    y='server_ttft_s',
    color='label',
    facet_col='in_tokens_label',
    color_discrete_map=COLOR_MAP,
    category_orders={
        'concurrency_str': CONC_STR,
        'in_tokens_label': INPUT_LABELS,
    },
    points='outliers',
    labels={
        'concurrency_str':  'Concurrency',
        'server_ttft_s':    'Server TTFT (s)',
        'label':            'Model',
        'in_tokens_label':  'Input Tokens',
    },
    title=(
        'Server TTFT Distribution per Concurrency — faceted by Input Tokens<br>'
        '<sub>Server-side distribution. At medium concurrency, encrypted may show LOWER server TTFT '
        'because TEE de-batching reduces the effective batch size seen by the server.</sub>'
    ),
)
fig.update_layout(height=450, width=1200, template='plotly_white')
fig.show()

In [ ]:
# Per-request: TEE network latency vs server TTFT (encrypted only), one column per Input Tokens
df_enc_ind = df_individual[
    (df_individual['label'] == 'Encrypted') &
    (~df_individual['is_error']) &
    (df_individual['server_network_latency_ms'].notna())
].copy()

fig = px.scatter(
    df_enc_ind,
    x='server_ttft_s',
    y='server_network_latency_ms',
    color='concurrency_str',
    facet_col='in_tokens_label',
    opacity=0.55,
    category_orders={
        'concurrency_str': CONC_STR,
        'in_tokens_label': INPUT_LABELS,
    },
    labels={
        'server_ttft_s':             'Server TTFT (s)',
        'server_network_latency_ms': 'TEE Network Latency (ms)',
        'in_tokens_label':           'Input Tokens',
        'concurrency_str':           'Concurrency',
    },
    title=(
        'Per-Request: TEE Network Latency vs Server TTFT — Encrypted (faceted by Input Tokens)<br>'
        '<sub>How network latency varies relative to server prefill time. '
        'A flatter horizontal spread = network latency is independent of server load.</sub>'
    ),
)
fig.update_layout(height=420, width=1300, template='plotly_white')
fig.show()

## 7. Summary

### Full Metric Comparison

In [ ]:
DIMS = ['num_input_tokens', 'num_output_tokens', 'num_concurrent_requests']

ALL_METRICS = [
    ('server_ttft_s_p50',               'Server TTFT p50 (s)'),
    ('server_output_token_per_s_p50',   'Server Tok/s p50'),
    ('server_end_to_end_latency_s_p50', 'Server E2E p50 (s)'),
    ('client_ttft_s_p50',               'Client TTFT p50 (s)'),
    ('client_end_to_end_latency_s_p50', 'Client E2E p50 (s)'),
    ('client_output_token_per_s_p50',   'Client Tok/s p50'),
    ('client_total_output_throughput',  'Total Throughput (tok/s)'),
    ('num_completed_requests_per_min',  'Req / min'),
    ('error_rate_pct',                  'Error Rate (%)'),
]

df_enc_s   = df_all[df_all['label'] == 'Encrypted'].set_index(DIMS)
df_plain_s = df_all[df_all['label'] == 'Non-Encrypted'].set_index(DIMS)

rows = []
for col, label in ALL_METRICS:
    if col not in df_enc_s.columns:
        continue
    for idx in df_enc_s.index:
        if idx not in df_plain_s.index:
            continue
        enc_v   = df_enc_s.loc[idx, col]
        plain_v = df_plain_s.loc[idx, col]
        if pd.isna(enc_v) and pd.isna(plain_v):
            continue
        delta   = enc_v - plain_v if pd.notna(enc_v) and pd.notna(plain_v) else np.nan
        pct_chg = delta / plain_v * 100 if pd.notna(delta) and plain_v != 0 else np.nan
        rows.append({
            'Metric':        label,
            'Input Tokens':  idx[0],
            'Output Tokens': idx[1],
            'Concurrency':   idx[2],
            'Non-Encrypted': plain_v,
            'Encrypted':     enc_v,
            'Delta':         delta,
            '% Change':      pct_chg,
        })

df_summary = pd.DataFrame(rows).sort_values(
    ['Metric', 'Input Tokens', 'Concurrency']
)

def _pct_color(val):
    if pd.isna(val): return ''
    if val >  100: return 'background-color: #ff6666'
    if val >   50: return 'background-color: #ffaaaa'
    if val >   20: return 'background-color: #ffd8aa'
    if val >    0: return 'background-color: #fffacc'
    if val < -20:  return 'background-color: #aaffaa'
    return ''

df_summary.style \
    .applymap(_pct_color, subset=['% Change']) \
    .format({
        'Non-Encrypted': '{:.3f}',
        'Encrypted':     '{:.3f}',
        'Delta':         '{:+.3f}',
        '% Change':      '{:+.1f}%',
    }, na_rep='—')

### TEE Overhead Attribution per Scenario

In [ ]:
DIMS = ['num_input_tokens', 'num_output_tokens', 'num_concurrent_requests']

df_tee_oh = df_all[df_all['label'] == 'Encrypted'].copy()
df_tee_oh = df_tee_oh.merge(
    df_all[df_all['label'] == 'Non-Encrypted'][DIMS + ['client_ttft_s_p50']].rename(
        columns={'client_ttft_s_p50': 'client_ttft_plain'}
    ),
    on=DIMS,
)

df_tee_oh['enc_time_s']    = df_tee_oh['total_encryption_time_ms_mean']  / 1000
df_tee_oh['dec_time_s']    = df_tee_oh['total_decryption_time_ms_mean']  / 1000
df_tee_oh['network_s']     = df_tee_oh['server_network_latency_ms_mean'] / 1000
df_tee_oh['tee_total_s']   = df_tee_oh['enc_time_s'] + df_tee_oh['dec_time_s'] + df_tee_oh['network_s']
df_tee_oh['ttft_delta_s']  = df_tee_oh['client_ttft_s_p50'] - df_tee_oh['client_ttft_plain']
df_tee_oh['network_pct']   = df_tee_oh['network_s']   / df_tee_oh['client_ttft_s_p50'] * 100
df_tee_oh['enc_pct']       = df_tee_oh['enc_time_s']  / df_tee_oh['client_ttft_s_p50'] * 100
df_tee_oh['dec_pct']       = df_tee_oh['dec_time_s']  / df_tee_oh['client_ttft_s_p50'] * 100
df_tee_oh['tee_total_pct'] = df_tee_oh['tee_total_s'] / df_tee_oh['client_ttft_s_p50'] * 100

DISPLAY = {
    'num_input_tokens':        'Input Tokens',
    'num_output_tokens':       'Output Tokens',
    'num_concurrent_requests': 'Concurrency',
    'client_ttft_plain':       'Plain TTFT (s)',
    'client_ttft_s_p50':       'Enc TTFT (s)',
    'ttft_delta_s':            'Delta (s)',
    'enc_time_s':              'Encryption (s)',
    'dec_time_s':              'Decryption (s)',
    'network_s':               'Network Hop (s)',
    'tee_total_s':             'TEE Total (s)',
    'tee_total_pct':           'TEE % of Enc TTFT',
    'network_pct':             'Network %',
    'enc_pct':                 'Encrypt %',
    'dec_pct':                 'Decrypt %',
}

df_tee_display = (
    df_tee_oh[list(DISPLAY.keys())]
    .rename(columns=DISPLAY)
    .sort_values(['Input Tokens', 'Concurrency'])
)

pct_cols = [c for c in df_tee_display.columns if '%' in c]
s_cols   = [c for c in df_tee_display.columns if '(s)' in c]

df_tee_display.style \
    .background_gradient(subset=['TEE % of Enc TTFT'], cmap='YlOrRd') \
    .format({c: '{:.1f}%' for c in pct_cols}) \
    .format({c: '{:.4f}'  for c in s_cols}, na_rep='—')